<a href="https://colab.research.google.com/github/Ziyi-star/Bachelorarbeit/blob/main/notebooks/training/train_1s_30hz_initial_2class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%load_ext autoreload
%autoreload 2

In [11]:
import os
import pickle
import scipy
import datetime
import numpy as np
import tensorflow as tf
import simclr_utitlities
import transformations
import simclr_models
import sys
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

sys.path.append('../../')   # Add parent directory to Python path
working_directory = "../models/"
sys.path.append('../../')
from utils.preprocessing import *

## Load data

In [5]:

with open('../../data/Field_validation/field_validation_segmented_data.pkl', 'rb') as f:
    loaded_data = pickle.load(f)

segments = loaded_data['segments']
segment_info = loaded_data['segment_info']

print(f"Loaded segments shape: {segments.shape}")
print(f"Loaded segment_info length: {len(segment_info)}")

Loaded segments shape: (2496, 30, 3)
Loaded segment_info length: 2496


In [7]:
df_segments_info = pd.DataFrame(segment_info)
df_segments_info.head(10)

,segment_id,start_time,end_time,start_index,curb_activity,curb_scene
0,0,2025-10-21 15:15:40.347,2025-10-21 15:15:41.304,0,0.0,0.0
1,1,2025-10-21 15:15:41.337,2025-10-21 15:15:42.294,30,0.0,0.0
2,2,2025-10-21 15:15:42.327,2025-10-21 15:15:43.284,60,0.0,0.0
3,3,2025-10-21 15:15:43.317,2025-10-21 15:15:44.274,90,0.0,0.0
4,4,2025-10-21 15:15:44.307,2025-10-21 15:15:45.264,120,0.0,0.0
5,5,2025-10-21 15:15:45.297,2025-10-21 15:15:46.254,150,0.0,0.0
6,6,2025-10-21 15:15:46.287,2025-10-21 15:15:47.244,180,0.0,0.0
7,7,2025-10-21 15:15:47.277,2025-10-21 15:15:48.234,210,0.0,0.0
8,8,2025-10-21 15:15:48.267,2025-10-21 15:15:49.224,240,0.0,0.0
9,9,2025-10-21 15:15:49.257,2025-10-21 15:15:50.214,270,0.0,0.0


In [9]:
# Extract curb_scene labels from segment_info
labels = np.array([info['curb_scene'] for info in segment_info])

print(f"Labels shape: {labels.shape}")
print(f"Unique labels: {np.unique(labels)}")


Labels shape: (2496,)
Unique labels: [0. 1.]


In [25]:
from tensorflow.keras.utils import to_categorical

# Convert labels to one-hot encoding
labels_one_hot = to_categorical(labels, num_classes=2)

print(f"Original labels shape: {labels.shape}")
print(f"One-hot labels shape: {labels_one_hot.shape}")
print(f"First 5 original labels: {labels[:5]}")
print(f"First 5 one-hot labels:\n{labels_one_hot[:5]}")

Original labels shape: (2496,)
One-hot labels shape: (2496, 2)
First 5 original labels: [0. 0. 0. 0. 0.]
First 5 one-hot labels:
[[1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]]


In [20]:
# Normalize segments
data_normalized = normalize_3d_data(segments)

## Load model and evaluate

### Model with pseudo-labels

In [26]:
full_eval_best_model_file_name = "../models/20251103-135527_simclr_full_eval_for_field_validation.keras"
full_eval_best_model = tf.keras.models.load_model(full_eval_best_model_file_name)

# Fix: Set is_one_hot=False since labels_one_hot contains integer labels, not one-hot encoded
print(simclr_utitlities.evaluate_model_simple(
    full_eval_best_model.predict(data_normalized), 
    labels_one_hot, 
    return_dict=True
))

 7/78 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step   

c:\Users\liuzi\miniconda3\Lib\site-packages\keras\src\models\functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input']
Received: inputs=Tensor(shape=(32, 30, 3))
  warnings.warn(msg)


78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
{'Confusion Matrix': array([[1248, 1190],
       [  12,   46]]), 'F1 Macro': 0.37302840501110507, 'F1 Micro': 0.5184294871794872, 'F1 Weighted': 0.6609273863428434, 'Precision': 0.5138465094775775, 'Recall': 0.6524992220870697, 'Kappa': np.float64(0.027946688231909378)}


In [30]:
# Analyze predictions and find optimal threshold
predictions = full_eval_best_model.predict(data_normalized)

print("Debug shapes:")
print(f"predictions shape: {predictions.shape}")
print(f"labels shape: {labels.shape}")

# Handle different prediction formats
if len(predictions.shape) > 1 and predictions.shape[1] == 2:
    # Binary classification with 2 output neurons - take the probability of class 1
    predictions_flat = predictions[:, 1]
elif len(predictions.shape) > 1 and predictions.shape[1] == 1:
    # Single output neuron
    predictions_flat = predictions.flatten()
else:
    # Already flattened
    predictions_flat = predictions.flatten()

print(f"predictions_flat shape after processing: {predictions_flat.shape}")
print(f"labels shape: {labels.shape}")

print("Prediction analysis:")
print(f"Mean prediction: {np.mean(predictions_flat):.3f}")
print(f"Std prediction: {np.std(predictions_flat):.3f}")
print(f"Min prediction: {np.min(predictions_flat):.3f}")
print(f"Max prediction: {np.max(predictions_flat):.3f}")

# Test different thresholds
thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
print("\nThreshold analysis:")
for thresh in thresholds:
    pred_binary = (predictions_flat > thresh).astype(int)
    fp = np.sum((labels == 0) & (pred_binary == 1))
    fn = np.sum((labels == 1) & (pred_binary == 0))
    precision = np.sum((labels == 1) & (pred_binary == 1)) / max(np.sum(pred_binary == 1), 1)
    recall = np.sum((labels == 1) & (pred_binary == 1)) / max(np.sum(labels == 1), 1)
    print(f"Threshold {thresh}: FP={fp:4d}, FN={fn:2d}, Precision={precision:.3f}, Recall={recall:.3f}")

78/78 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Debug shapes:
predictions shape: (2496, 2)
labels shape: (2496,)
predictions_flat shape after processing: (2496,)
labels shape: (2496,)
Prediction analysis:
Mean prediction: 0.489
Std prediction: 0.398
Min prediction: 0.000
Max prediction: 1.000

Threshold analysis:
Threshold 0.5: FP=1190, FN=12, Precision=0.037, Recall=0.793
Threshold 0.6: FP=1086, FN=12, Precision=0.041, Recall=0.793
Threshold 0.7: FP= 970, FN=13, Precision=0.044, Recall=0.776
Threshold 0.8: FP= 824, FN=17, Precision=0.047, Recall=0.707
Threshold 0.9: FP= 660, FN=25, Precision=0.048, Recall=0.569


### Model without pseudo labels

In [31]:
full_eval_best_model_real_world_file_name = "../models/20251102-141233_simclr_full_eval_1s_30hz_for_real_world.keras"
full_eval_best_model = tf.keras.models.load_model(full_eval_best_model_real_world_file_name)

# Fix: Set is_one_hot=False since labels_one_hot contains integer labels, not one-hot encoded
print(simclr_utitlities.evaluate_model_simple(
    full_eval_best_model.predict(data_normalized), 
    labels_one_hot, 
    return_dict=True
))

 8/78 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step   

c:\Users\liuzi\miniconda3\Lib\site-packages\keras\src\models\functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input']
Received: inputs=Tensor(shape=(32, 30, 3))
  warnings.warn(msg)


78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
{'Confusion Matrix': array([[1464,  974],
       [  15,   43]]), 'F1 Macro': 0.4137554250702068, 'F1 Micro': 0.6037660256410257, 'F1 Weighted': 0.7319997807060611, 'Precision': 0.5160696157213776, 'Recall': 0.6709357585358264, 'Kappa': np.float64(0.03769021700995001)}


In [32]:
# Load and evaluate the real world model with different thresholds
full_eval_best_model_real_world_file_name = "../models/20251102-141233_simclr_full_eval_1s_30hz_for_real_world.keras"
full_eval_best_model_real_world = tf.keras.models.load_model(full_eval_best_model_real_world_file_name)

# Get predictions from the real world model
predictions_real_world = full_eval_best_model_real_world.predict(data_normalized)

print("Real World Model - Debug shapes:")
print(f"predictions shape: {predictions_real_world.shape}")
print(f"labels shape: {labels.shape}")

# Handle different prediction formats
if len(predictions_real_world.shape) > 1 and predictions_real_world.shape[1] == 2:
    # Binary classification with 2 output neurons - take the probability of class 1
    predictions_real_world_flat = predictions_real_world[:, 1]
elif len(predictions_real_world.shape) > 1 and predictions_real_world.shape[1] == 1:
    # Single output neuron
    predictions_real_world_flat = predictions_real_world.flatten()
else:
    # Already flattened
    predictions_real_world_flat = predictions_real_world.flatten()

print(f"predictions_flat shape after processing: {predictions_real_world_flat.shape}")

print("Real World Model - Prediction analysis:")
print(f"Mean prediction: {np.mean(predictions_real_world_flat):.3f}")
print(f"Std prediction: {np.std(predictions_real_world_flat):.3f}")
print(f"Min prediction: {np.min(predictions_real_world_flat):.3f}")
print(f"Max prediction: {np.max(predictions_real_world_flat):.3f}")

# Test different thresholds for the real world model
thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
print("\nReal World Model - Threshold analysis:")
for thresh in thresholds:
    pred_binary = (predictions_real_world_flat > thresh).astype(int)
    fp = np.sum((labels == 0) & (pred_binary == 1))
    fn = np.sum((labels == 1) & (pred_binary == 0))
    tp = np.sum((labels == 1) & (pred_binary == 1))
    tn = np.sum((labels == 0) & (pred_binary == 0))
    
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    accuracy = (tp + tn) / len(labels)
    
    print(f"Threshold {thresh}: FP={fp:4d}, FN={fn:2d}, TP={tp:2d}, TN={tn:4d}, Precision={precision:.3f}, Recall={recall:.3f}, Accuracy={accuracy:.3f}")

18/78 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step   

c:\Users\liuzi\miniconda3\Lib\site-packages\keras\src\models\functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input']
Received: inputs=Tensor(shape=(32, 30, 3))
  warnings.warn(msg)


78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Real World Model - Debug shapes:
predictions shape: (2496, 2)
labels shape: (2496,)
predictions_flat shape after processing: (2496,)
Real World Model - Prediction analysis:
Mean prediction: 0.419
Std prediction: 0.387
Min prediction: 0.000
Max prediction: 1.000

Real World Model - Threshold analysis:
Threshold 0.5: FP= 974, FN=15, TP=43, TN=1464, Precision=0.042, Recall=0.741, Accuracy=0.604
Threshold 0.6: FP= 860, FN=18, TP=40, TN=1578, Precision=0.044, Recall=0.690, Accuracy=0.648
Threshold 0.7: FP= 766, FN=19, TP=39, TN=1672, Precision=0.048, Recall=0.672, Accuracy=0.685
Threshold 0.8: FP= 643, FN=21, TP=37, TN=1795, Precision=0.054, Recall=0.638, Accuracy=0.734
Threshold 0.9: FP= 501, FN=27, TP=31, TN=1937, Precision=0.058, Recall=0.534, Accuracy=0.788
